In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, BooleanType
from pyspark.sql.functions import col, when, concat, lit, rank, sum
from pyspark.sql.window import Window

flight_schema = StructType([
    StructField("flight_id", StringType(), True),
    StructField("airline", StringType(), True),
    StructField("from_city", StringType(), True),
    StructField("to_city", StringType(), True),
    StructField("duration", IntegerType(), True),
    StructField("status", StringType(), True)
])

booking_schema = StructType([
    StructField("booking_id", StringType(), True),
    StructField("flight_id", StringType(), True),
    StructField("passenger_name", StringType(), True),
    StructField("travel_class", StringType(), True),
    StructField("ticket_price", IntegerType(), True),
    StructField("booking_date", StringType(), True)
])

pref_schema = StructType([
    StructField("passenger_name", StringType(), True),
    StructField("meal", StringType(), True),
    StructField("seat", StringType(), True),
    StructField("extra_baggage", BooleanType(), True)
])

flights_data = [
    ("F101", "Indigo", "Hyderabad", "Delhi", 140, "On Time"),
    ("F102", "Air India", "Mumbai", "Chennai", 120, "Delayed"),
    ("F103", "Vistara", "Bangalore", "Hyderabad", 90, "On Time"),
    ("F104", "Indigo", "Delhi", "Mumbai", 130, "Cancelled"),
    ("F105", "Air India", "Chennai", "Bangalore", 80, "On Time"),
    ("F106", "Akasa", "Pune", "Delhi", 150, "Delayed"),
    ("F107", "Vistara", "Hyderabad", "Kolkata", 160, "On Time"),
    ("F108", "Indigo", "Mumbai", "Hyderabad", 110, "On Time"),
    ("F109", "Akasa", "Delhi", "Chennai", 145, "Delayed"),
    ("F110", "Air India", "Bangalore", "Mumbai", 95, "On Time"),
    ("F111", "Indigo", "Hyderabad", "Goa", 75, "On Time"),
    ("F112", "Vistara", "Goa", "Delhi", 150, "Cancelled"),
    ("F113", "Akasa", "Chennai", "Pune", 100, "On Time"),
    ("F114", "Air India", "Kolkata", "Bangalore", 170, "Delayed"),
    ("F115", "Indigo", "Delhi", "Hyderabad", 135, "On Time"),
]

bookings_data = [
    ("B1001", "F101", "Rahul Sharma", "Economy", 8500, "2026-06-01"),
    ("B1002", "F101", "Priya Reddy", "Business", 22000, "2026-06-01"),
    ("B1003", "F102", "Amit Kumar", "Economy", 9000, "2026-06-02"),
    ("B1004", "F103", "Sneha Patel", "Premium Economy", 15000, "2026-06-02"),
    ("B1005", "F104", "Farhan Ali", "Economy", 7500, "2026-06-03"),
    ("B1006", "F105", "Neha Singh", "Business", 25000, "2026-06-03"),
    ("B1007", "F106", "Arjun Verma", "Economy", 10000, "2026-06-04"),
    ("B1008", "F107", "Meera Nair", "Premium Economy", 17000, "2026-06-04"),
    ("B1009", "F108", "Kiran Rao", "Economy", 9500, "2026-06-05"),
    ("B1010", "F109", "Nisha Reddy", "Business", 28000, "2026-06-05"),
    ("B1011", "F110", "David Thomas", "Economy", 8000, "2026-06-06"),
    ("B1012", "F111", "Ayesha Khan", "Premium Economy", 16000, "2026-06-06"),
    ("B1013", "F112", "Rohit Sharma", "Economy", 7000, "2026-06-07"),
    ("B1014", "F113", "Pooja Mehta", "Business", 24000, "2026-06-07"),
    ("B1015", "F114", "Sanjay Gupta", "Economy", 10500, "2026-06-08"),
    ("B1016", "F115", "Divya Iyer", "Premium Economy", 18000, "2026-06-08"),
    ("B1017", "F101", "Rahul Sharma", "Economy", 8500, "2026-06-09"),
    ("B1018", "F103", "Priya Reddy", "Business", 23000, "2026-06-09"),
    ("B1019", "F107", "Amit Kumar", "Economy", 9500, "2026-06-10"),
    ("B1020", "F110", "Sneha Patel", "Premium Economy", 15500, "2026-06-10"),
]

prefs_data = [
    ("Rahul Sharma", "Veg", "Window", True),
    ("Priya Reddy", "Non-Veg", "Aisle", False),
    ("Amit Kumar", "Veg", "Middle", False),
    ("Sneha Patel", "Jain", "Window", True),
    ("Farhan Ali", "Non-Veg", "Aisle", False),
    ("Neha Singh", "Veg", "Window", True),
    ("Arjun Verma", "Veg", "Middle", False),
    ("Meera Nair", "Jain", "Window", True),
    ("Kiran Rao", "Veg", "Aisle", False),
    ("Nisha Reddy", "Non-Veg", "Window", True),
    ("David Thomas", "Veg", "Middle", False),
    ("Ayesha Khan", "Jain", "Window", True),
    ("Rohit Sharma", "Veg", "Aisle", False),
    ("Pooja Mehta", "Non-Veg", "Window", True),
    ("Sanjay Gupta", "Veg", "Middle", False),
    ("Divya Iyer", "Jain", "Window", True),
]

df_flights = spark.createDataFrame(flights_data, flight_schema)
df_bookings = spark.createDataFrame(bookings_data, booking_schema)
df_prefs = spark.createDataFrame(prefs_data, pref_schema)
df_joined = df_bookings.join(df_flights, on="flight_id", how="left")
df_transformed = df_joined.withColumn("revenue", col("ticket_price")) \
    .withColumn("delay_flag", when(col("status") == "Delayed", "Yes").otherwise("No"))
df_final = df_transformed.join(df_prefs, on="passenger_name", how="left")
df_final.createOrReplaceTempView("final_data")

In [0]:
print("Airline Revenue Report")
display(spark.sql("""
    SELECT airline AS Airline, SUM(revenue) AS Revenue
    FROM final_data
    GROUP BY airline
    ORDER BY Revenue DESC
"""))

Airline Revenue Report


Airline,Revenue
Indigo,90000
Vistara,71500
Air India,68000
Akasa,62000


In [0]:
print("Route Performance Report")
display(spark.sql("""
    SELECT concat(from_city, ' -> ', to_city) AS Route, SUM(revenue) AS Revenue
    FROM final_data
    GROUP BY from_city, to_city
    ORDER BY Revenue DESC
"""))

Route Performance Report


Route,Revenue
Hyderabad -> Delhi,39000
Bangalore -> Hyderabad,38000
Delhi -> Chennai,28000
Hyderabad -> Kolkata,26500
Chennai -> Bangalore,25000
Chennai -> Pune,24000
Bangalore -> Mumbai,23500
Delhi -> Hyderabad,18000
Hyderabad -> Goa,16000
Kolkata -> Bangalore,10500


In [0]:
print("Passenger Preference Report")
display(spark.sql("""
    SELECT passenger_name AS Passenger, meal AS Meal, seat AS Seat
    FROM final_data
    GROUP BY passenger_name, meal, seat
    ORDER BY passenger_name
"""))

Passenger Preference Report


Passenger,Meal,Seat
Amit Kumar,Veg,Middle
Arjun Verma,Veg,Middle
Ayesha Khan,Jain,Window
David Thomas,Veg,Middle
Divya Iyer,Jain,Window
Farhan Ali,Non-Veg,Aisle
Kiran Rao,Veg,Aisle
Meera Nair,Jain,Window
Neha Singh,Veg,Window
Nisha Reddy,Non-Veg,Window


In [0]:
print("Flight Delay Report")
display(spark.sql("""
    SELECT flight_id AS Flight, status AS Status
    FROM final_data
    GROUP BY flight_id, status
    ORDER BY flight_id
"""))

Flight Delay Report


Flight,Status
F101,On Time
F102,Delayed
F103,On Time
F104,Cancelled
F105,On Time
F106,Delayed
F107,On Time
F108,On Time
F109,Delayed
F110,On Time


In [0]:
print("Top Revenue Flights")
window_rank = Window.orderBy(col("Revenue").desc())
df_top = spark.sql("""
    SELECT flight_id AS Flight, SUM(revenue) AS Revenue
    FROM final_data
    GROUP BY flight_id
""").withColumn("Rank", rank().over(window_rank))
display(df_top.orderBy("Rank"))

Top Revenue Flights


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Flight,Revenue,Rank
F101,39000,1
F103,38000,2
F109,28000,3
F107,26500,4
F105,25000,5
F113,24000,6
F110,23500,7
F115,18000,8
F111,16000,9
F114,10500,10
